# SafeDrive CU03 — Pipeline completo

Ejecuta las secciones **en orden**:
1. Genera dataset sintético
2. Procesa dataset de Kaggle
3. Procesa datos locales (Room / Android)
4. Monta el dataset final
5. Entrena y exporta `.keras` + `.tflite`

In [ ]:
# Dependencias comunes a todo el notebook
import os
import glob
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split

print(f'TensorFlow {tf.__version__}')

---
## 1. Genera dataset sintético
Genera datos físicamente realistas para las 3 clases.

**Salida:** `ml/data/processed/source/datos_sinteticos.csv`

In [ ]:
for folder in ['ml/data/processed/source', 'ml/data/processed/dataset/wrong_data', 'models']:
    os.makedirs(folder, exist_ok=True)

def generar_datos(n_samples, label):
    if label == 0:   # Falso Positivo — conducción normal
        peak_g    = np.random.uniform(0.1,  1.5, n_samples)
        jerk      = np.random.uniform(0.1,  0.8, n_samples)
        audio     = np.random.uniform(0.0,  0.2, n_samples)
        angular   = np.random.uniform(0.0,  0.3, n_samples)
        velocidad = np.random.uniform(10,  120,  n_samples)
    elif label == 1: # Posible Susto — frenazo / derrape
        peak_g    = np.random.uniform(1.5,  4.0, n_samples)
        jerk      = np.random.uniform(0.8,  2.5, n_samples)
        audio     = np.random.uniform(0.1,  0.4, n_samples)
        angular   = np.random.uniform(0.3,  0.8, n_samples)
        velocidad = np.random.uniform(20,  100,  n_samples)
    elif label == 2: # Choque Grave
        peak_g    = np.random.uniform(4.0, 12.0, n_samples)
        jerk      = np.random.uniform(2.5,  8.0, n_samples)
        audio     = np.random.uniform(0.6,  1.0, n_samples)
        angular   = np.random.uniform(0.5,  2.0, n_samples)
        velocidad = np.random.uniform(30,  120,  n_samples)

    return pd.DataFrame({
        'Peak_G':         peak_g,
        'Jerk':           jerk,
        'Firma_Acustica': audio,
        'Cambio_Angular': angular,
        'Velocidad':      velocidad,
        'Label':          [label] * n_samples
    })

df_sintetico = pd.concat([
    generar_datos(4000, 0),
    generar_datos(1500, 1),
    generar_datos(1500, 2)
]).sample(frac=1, random_state=42).reset_index(drop=True)

# Dataset basura (outliers físicamente imposibles)
wrong_data = pd.DataFrame({
    'Peak_G':         [500.0, -2.0,   0.0,  999.9],
    'Jerk':           [0.0,  900.0, -10.0,    0.0],
    'Firma_Acustica': [2.5,   -0.5,   0.0,   10.0],
    'Cambio_Angular': [0.0,    0.0,   0.0,   50.0],
    'Velocidad':      [-50,    400,     0,      0],
    'Label':          [0,       1,     9,      2]
})

df_sintetico.to_csv('ml/data/processed/source/datos_sinteticos.csv', index=False)
wrong_data.to_csv('ml/data/processed/dataset/wrong_data/corrupted_data.csv', index=False)

print(f'Sintético: {len(df_sintetico)} filas')
print(df_sintetico['Label'].value_counts().sort_index())

---
## 2. Procesa dataset de Kaggle
Extrae features del CSV crudo y re-etiqueta los impactos como `Susto (1)` ya que Kaggle solo registra ~0.47G.

**Requiere:** `ml/data/raw/road_accident_imu_dataset_8000.csv`

**Salida:** `ml/data/processed/source/kaggle_data.csv`

In [ ]:
input_kaggle  = 'ml/data/raw/road_accident_imu_dataset_8000.csv'
output_kaggle = 'ml/data/processed/source/kaggle_data.csv'

if not os.path.exists(input_kaggle):
    print(f'AVISO: No se encuentra {input_kaggle}. Omitiendo sección Kaggle.')
else:
    df_raw = pd.read_csv(input_kaggle)

    # Extracción de características
    df_raw['Peak_G']         = abs((df_raw['Motion_Intensity'] / 9.8) - 1.0)
    df_raw['Jerk']           = df_raw['Peak_G'].diff().abs().fillna(0)
    df_raw['Cambio_Angular'] = np.sqrt(df_raw['Gyro_X']**2 + df_raw['Gyro_Y']**2 + df_raw['Gyro_Z']**2)
    df_raw['Velocidad']      = df_raw['Speed_kmh']

    # Firma acústica sintética (Kaggle no tiene audio)
    np.random.seed(42)
    df_raw['Firma_Acustica'] = np.where(
        df_raw['Crash_Label'] == 1,
        np.random.uniform(0.1, 0.4, len(df_raw)),  # Susto
        np.random.uniform(0.0, 0.1, len(df_raw))   # Normal
    )

    # Re-etiquetado: impactos Kaggle son ~0.47G -> Susto (1), no Choque Grave (2)
    def mapear_etiquetas(row):
        if row['Crash_Label'] == 1: return 1
        if row['Peak_G'] > 1.0:     return 1
        return 0

    df_raw['Label'] = df_raw.apply(mapear_etiquetas, axis=1)

    df_kaggle = df_raw[['Peak_G', 'Jerk', 'Firma_Acustica', 'Cambio_Angular', 'Velocidad', 'Label']]
    df_kaggle.to_csv(output_kaggle, index=False)

    print(f'Kaggle: {len(df_kaggle)} filas procesadas -> {output_kaggle}')
    print(df_kaggle['Label'].value_counts().sort_index())

---
## 3. Procesa datos locales (Room / Android)
Normaliza los nombres de columnas exportados desde Room al estándar SafeDrive.
Usa el feedback del usuario como etiqueta si existe; si no, aplica lógica matemática.

**Requiere:** `ml/data/raw/mis_datos_room.csv`

**Salida:** `ml/data/processed/source/local_room_data.csv`

In [ ]:
input_local  = 'ml/data/raw/mis_datos_room.csv'
output_local = 'ml/data/processed/source/local_room_data.csv'

if not os.path.exists(input_local):
    print(f'AVISO: No se encuentra {input_local}. Omitiendo sección local.')
else:
    df_raw_local = pd.read_csv(input_local)
    df_local = pd.DataFrame()

    # Mapeo Room -> estándar SafeDrive (ajusta los nombres si cambian en Kotlin)
    df_local['Peak_G']         = df_raw_local['max_g_force']
    df_local['Velocidad']      = df_raw_local['speed_kmh']
    df_local['Firma_Acustica'] = df_raw_local['audio_level']
    df_local['Jerk'] = (
        df_raw_local['jerk'] if 'jerk' in df_raw_local.columns
        else df_local['Peak_G'] * 0.7
    )
    df_local['Cambio_Angular'] = (
        df_raw_local['gyro_magnitude'] if 'gyro_magnitude' in df_raw_local.columns
        else df_local['Peak_G'] * 0.1
    )

    # Etiquetado: feedback del usuario > lógica matemática
    if 'user_confirmation' in df_raw_local.columns:
        df_local['Label'] = df_raw_local['user_confirmation']
    else:
        def logica_matematica(row):
            if row['Peak_G'] > 4.0: return 2
            if row['Peak_G'] > 1.5: return 1
            return 0
        df_local['Label'] = df_local.apply(logica_matematica, axis=1)

    df_local.to_csv(output_local, index=False)
    print(f'Local: {len(df_local)} registros procesados -> {output_local}')
    print(df_local['Label'].value_counts().sort_index())

---
## 4. Monta el dataset final
Fusiona todas las fuentes disponibles en `ml/data/processed/source/`, mezcla y divide en **train 70% / validation 15% / test 15%**.

In [ ]:
for folder in ['ml/data/processed/dataset/train',
               'ml/data/processed/dataset/validation',
               'ml/data/processed/dataset/test']:
    os.makedirs(folder, exist_ok=True)

# Leer todas las fuentes (wrong_data queda excluido por estar en otra carpeta)
archivos = glob.glob('ml/data/processed/source/*.csv')
print(f'Fuentes encontradas ({len(archivos)}):')
for f in archivos:
    df_tmp = pd.read_csv(f)
    print(f'  {f}  ->  {len(df_tmp)} filas')

dataset_completo = pd.concat(
    [pd.read_csv(f) for f in archivos], ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'\nTotal fusionado: {len(dataset_completo)} filas')
print(f'Distribución de clases:\n{dataset_completo["Label"].value_counts().sort_index()}')

# Split 70 / 15 / 15
df_train, df_temp = train_test_split(dataset_completo, test_size=0.30, random_state=42)
df_val,   df_test = train_test_split(df_temp,          test_size=0.50, random_state=42)

df_train.to_csv('ml/data/processed/dataset/train/train.csv',         index=False)
df_val.to_csv('ml/data/processed/dataset/validation/validation.csv', index=False)
df_test.to_csv('ml/data/processed/dataset/test/test.csv',            index=False)

print(f'\nTrain: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}')

---
## 5. Entrena y exporta modelos
Entrena la red neuronal y exporta dos formatos:
- **`.keras`** — para reentrenar o inspeccionar en Python
- **`.tflite`** — para desplegar en el dispositivo Android

In [ ]:
# Cargar los splits generados en la sección anterior
df_train = pd.read_csv('ml/data/processed/dataset/train/train.csv')
df_val   = pd.read_csv('ml/data/processed/dataset/validation/validation.csv')
df_test  = pd.read_csv('ml/data/processed/dataset/test/test.csv')

X_train = df_train.drop(columns=['Label']).values;  y_train = df_train['Label'].values
X_val   = df_val.drop(columns=['Label']).values;    y_val   = df_val['Label'].values
X_test  = df_test.drop(columns=['Label']).values;   y_test  = df_test['Label'].values

n_features = X_train.shape[1]
print(f'Features: {n_features} | Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

In [ ]:
# Definir modelo (input_shape dinámico según features del dataset)
model = keras.Sequential([
    keras.layers.Dense(24, activation='relu', input_shape=(n_features,)),
    keras.layers.Dense(12, activation='relu'),
    keras.layers.Dense(3,  activation='softmax')
], name='safedrive_cu03')

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=40,
    batch_size=16,
    validation_data=(X_val, y_val)
)

In [ ]:
# Evaluación sobre test
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Test Loss:     {loss:.4f}')
print(f'Test Accuracy: {accuracy:.4f}')

clases = ['Falso Positivo', 'Posible Susto', 'Choque Grave']
predicciones = model.predict(X_test)
print('\nPrimeras 10 predicciones:')
for i in range(min(10, len(y_test))):
    idx  = np.argmax(predicciones[i])
    real = int(y_test[i])
    marca = 'OK' if idx == real else 'X'
    print(f'  [{marca}]  Pred: {clases[idx]:20s} | Real: {clases[real]}')

In [ ]:
os.makedirs('models', exist_ok=True)

# Formato Keras — para reentrenar o inspeccionar
keras_path = 'models/safedrive_cu03_model.keras'
model.save(keras_path)
print(f'Keras  guardado en: {keras_path}  ({os.path.getsize(keras_path)/1024:.1f} KB)')

# Formato TFLite — para desplegar en Android
tflite_path = 'models/safedrive_cu03_model.tflite'
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)
print(f'TFLite guardado en: {tflite_path}  ({os.path.getsize(tflite_path)/1024:.1f} KB)')